## Date Handling of Input Files

> The data contains periods in CET/CEST format.
	

* we need to have start and end date in one consitent time zone
* we need to split the period into start and end date of the period 

"01/01/2015 00:00:00 - 01/01/2015 00:15:00"


## Setup

All the manipulations and plots in this notebook can be created with standard libraries such as matplotlib, statsmodels etc. 

In [96]:
# Main data packages. 
import numpy as np
import pandas as pd


## Import Data 

The data for this notebook was downloaded from the [meteoblue website](https://www.meteoblue.com/en/weather/archive/export/basel_switzerland_2661604) and consits of weather data for the city of Basel from 2008 till 2020. 

In [97]:

df_core = pd.read_csv("../../data_cleaned/merged/Data_imputed_2019_to_2025_with_Frourier_and_holidays.csv", delimiter=",")


In [98]:
df_core.shape

(61367, 34)

In [99]:
df_price_gas = pd.read_csv("../../data_cleaned/by_source/07_prices_gas.csv", delimiter=",")

In [100]:
df_price_gas.shape

(2016, 3)

In [101]:
df_core.columns

Index(['date', 'year', 'month', 'day', 'dayofyear', 'hour', 'week',
       'dayofweek', 'price', 'period_start_utc', 'period_end_utc', 'c_by_hour',
       'load_forecast_da', 'load_actual', 'off_wind_da', 'off_wind_act',
       'on_wind_da', 'on_wind_act', 'solar_da', 'solar_act', 'gen_forecast_da',
       'gen_actual', 'res_sum_da', 'res_sum_act', 'imputed', 'interpolated',
       'dayofyear_sin1', 'dayofyear_cos1', 'hour_sin1', 'hour_cos1',
       'dayofweek_sin1', 'dayofweek_cos1', 'is_holiday', 'day_type'],
      dtype='object')

In [102]:
group_cols = [
    "date", "year", "month", "day", "dayofyear", 
    #"week", 
    "dayofweek",'dayofyear_sin1', 'dayofyear_cos1', #'hour_sin1', 'hour_cos1',
       'dayofweek_sin1', 'dayofweek_cos1', 'is_holiday', 'day_type'
]

df_core_d = (
    df_core
    .groupby(group_cols, as_index=False)
    .agg(
        period_start_utc=("period_start_utc", "min"),
        period_end_utc=("period_end_utc", "max"),
        price=("price", "mean"),
        gen_forecast_da=("load_forecast_da", "mean"),
        load_forecast_da=("gen_forecast_da", "mean"),
        on_wind_da=("on_wind_da", "mean"),
        off_wind_da=("off_wind_da", "mean"),
        solar_da=("solar_da", "mean"),
        res_sum_da=("res_sum_da", "mean"),
        c_by_day=("year", "size"),   # count rows per group
    )
)
df_core_d.head()

,date,year,month,day,dayofyear,dayofweek,dayofyear_sin1,dayofyear_cos1,dayofweek_sin1,dayofweek_cos1,...,period_start_utc,period_end_utc,price,gen_forecast_da,load_forecast_da,on_wind_da,off_wind_da,solar_da,res_sum_da,c_by_day
0,2019-01-01,2019,1,1,1,1,0.000000,1.000000,0.781831,0.623490,...,2019-01-01 00:00:00+00:00,2019-01-02 00:00:00+00:00,-5.873333,47744.070208,61009.409167,31139.542708,4523.792188,413.402396,36076.737292,24
1,2019-01-02,2019,1,2,2,2,0.017202,0.999852,0.974928,-0.222521,...,2019-01-02 00:00:00+00:00,2019-01-03 00:00:00+00:00,29.286250,55944.564583,65324.240833,24192.978333,4118.932917,1191.025417,29502.936667,24
2,2019-01-03,2019,1,3,3,3,0.034398,0.999408,0.433884,-0.900969,...,2019-01-03 00:00:00+00:00,2019-01-04 00:00:00+00:00,58.197917,57691.780937,64042.930417,10048.293333,2879.674792,942.652292,13870.620417,24
3,2019-01-04,2019,1,4,4,4,0.051584,0.998669,-0.433884,-0.900969,...,2019-01-04 00:00:00+00:00,2019-01-05 00:00:00+00:00,49.440000,55457.005625,68236.983750,17837.964792,4346.913333,375.754687,22560.632812,24
4,2019-01-05,2019,1,5,5,5,0.068755,0.997634,-0.974928,-0.222521,...,2019-01-05 00:00:00+00:00,2019-01-06 00:00:00+00:00,43.214583,52843.881458,62672.755417,19964.511042,3769.793021,261.317813,23995.621875,24


In [103]:
df_core_d["c_by_day"].value_counts()

c_by_day
24    2556
23       1
Name: count, dtype: int64

In [104]:
df_core['date'].head(2)

0    2019-01-01
1    2019-01-01
Name: date, dtype: object

In [105]:
df_price_gas.tail(2)

,date,price_gas_open,price_gas_close
2014,2018-01-03,19.325,19.325
2015,2018-01-02,19.320,19.320


In [106]:
from functools import reduce

#dfs = [df_price, df_res_offshore] ##, df_res_offshore, df_res_onshore, df_res_solar]  investigate on keys df_gen_forecast
dfs_d = [
    df_core_d,
    df_price_gas
    # df_load[["period_start_utc","load_forecast_da","load_actual"]],  
]

df_merged_d = reduce(
    lambda left, right: pd.merge(left, right, on="date", how="left"),
    dfs_d
)

In [107]:
df_merged_d.shape

(2557, 24)

In [108]:
df_merged_d.head(5)

,date,year,month,day,dayofyear,dayofweek,dayofyear_sin1,dayofyear_cos1,dayofweek_sin1,dayofweek_cos1,...,price,gen_forecast_da,load_forecast_da,on_wind_da,off_wind_da,solar_da,res_sum_da,c_by_day,price_gas_open,price_gas_close
0,2019-01-01,2019,1,1,1,1,0.000000,1.000000,0.781831,0.623490,...,-5.873333,47744.070208,61009.409167,31139.542708,4523.792188,413.402396,36076.737292,24,NaN,NaN
1,2019-01-02,2019,1,2,2,2,0.017202,0.999852,0.974928,-0.222521,...,29.286250,55944.564583,65324.240833,24192.978333,4118.932917,1191.025417,29502.936667,24,22.280,22.475
2,2019-01-03,2019,1,3,3,3,0.034398,0.999408,0.433884,-0.900969,...,58.197917,57691.780937,64042.930417,10048.293333,2879.674792,942.652292,13870.620417,24,22.215,22.255
3,2019-01-04,2019,1,4,4,4,0.051584,0.998669,-0.433884,-0.900969,...,49.440000,55457.005625,68236.983750,17837.964792,4346.913333,375.754687,22560.632812,24,22.845,22.930
4,2019-01-05,2019,1,5,5,5,0.068755,0.997634,-0.974928,-0.222521,...,43.214583,52843.881458,62672.755417,19964.511042,3769.793021,261.317813,23995.621875,24,NaN,NaN


### now filling in NaN for gas_prices, since there is no data on weekends and on holidays

#### 1. fill the new column 'gas_price' with 'gas_price_open'. If this is a NaN value, take the 'price_gas_close' from the day before, if available.
#### 2 for any remaining value in 'price_gas' interpolate using the surrounding values


In [109]:
df = df_merged_d.copy()
# Ensure chronological order
df = df.sort_values("period_start_utc").copy()

# 0) Prepare day key
#df["date"] = pd.to_datetime(df["period_start_utc"]).dt.normalize()

# 1) Start with gas_price_open
df["price_gas"] = df["price_gas_open"]

# If gas_price is NaN, use previous day's price_gas_close (if available)
close_by_day = df.groupby("date")["price_gas_close"].last()   # daily close
prev_day_close = df["date"].map(close_by_day.shift(1))
df["price_gas"] = df["price_gas"].fillna(prev_day_close)

# 2) Interpolate remaining NaNs using surrounding values
df["price_gas"] = df["price_gas"].interpolate(method="linear", limit_direction="both")

# Optional cleanup
##df = df.drop(columns=["date"])


In [110]:
df.columns

Index(['date', 'year', 'month', 'day', 'dayofyear', 'dayofweek',
       'dayofyear_sin1', 'dayofyear_cos1', 'dayofweek_sin1', 'dayofweek_cos1',
       'is_holiday', 'day_type', 'period_start_utc', 'period_end_utc', 'price',
       'gen_forecast_da', 'load_forecast_da', 'on_wind_da', 'off_wind_da',
       'solar_da', 'res_sum_da', 'c_by_day', 'price_gas_open',
       'price_gas_close', 'price_gas'],
      dtype='object')

In [111]:
# Check NaN only in one column ('price')
rows_with_missing_price = df[df["price_gas"].isna()]
rows_with_missing_price.tail(100)

,date,year,month,day,dayofyear,dayofweek,dayofyear_sin1,dayofyear_cos1,dayofweek_sin1,dayofweek_cos1,...,gen_forecast_da,load_forecast_da,on_wind_da,off_wind_da,solar_da,res_sum_da,c_by_day,price_gas_open,price_gas_close,price_gas


In [112]:
df = df.drop(columns=["price_gas_open","price_gas_close"])
df_d = df.copy()
df_d.to_csv("../../data_cleaned/merged/02_4_clean_data_daily_mean.csv", index=False)

In [113]:
dfs_h = [
    df_core,
    df_d[["date","price_gas"]],  
]

df_merged_h = reduce(
    lambda left, right: pd.merge(left, right, on="date", how="left"),
    dfs_h
)



In [114]:
df_merged_h.shape

(61367, 35)

In [115]:
df_merged_h.head()

,date,year,month,day,dayofyear,hour,week,dayofweek,price,period_start_utc,...,interpolated,dayofyear_sin1,dayofyear_cos1,hour_sin1,hour_cos1,dayofweek_sin1,dayofweek_cos1,is_holiday,day_type,price_gas
0,2019-01-01,2019,1,1,1,0,1,1,10.07,2019-01-01 00:00:00+00:00,...,0,0.0,1.0,0.000000,1.000000,0.781831,0.62349,1,holiday,22.28
1,2019-01-01,2019,1,1,1,1,1,1,-4.08,2019-01-01 01:00:00+00:00,...,0,0.0,1.0,0.258819,0.965926,0.781831,0.62349,1,holiday,22.28
2,2019-01-01,2019,1,1,1,2,1,1,-9.91,2019-01-01 02:00:00+00:00,...,0,0.0,1.0,0.500000,0.866025,0.781831,0.62349,1,holiday,22.28
3,2019-01-01,2019,1,1,1,3,1,1,-7.41,2019-01-01 03:00:00+00:00,...,0,0.0,1.0,0.707107,0.707107,0.781831,0.62349,1,holiday,22.28
4,2019-01-01,2019,1,1,1,4,1,1,-12.55,2019-01-01 04:00:00+00:00,...,0,0.0,1.0,0.866025,0.500000,0.781831,0.62349,1,holiday,22.28


In [116]:
df_merged_h.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61367 entries, 0 to 61366
Data columns (total 35 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   date              61367 non-null  object 
 1   year              61367 non-null  int64  
 2   month             61367 non-null  int64  
 3   day               61367 non-null  int64  
 4   dayofyear         61367 non-null  int64  
 5   hour              61367 non-null  int64  
 6   week              61367 non-null  int64  
 7   dayofweek         61367 non-null  int64  
 8   price             61367 non-null  float64
 9   period_start_utc  61367 non-null  object 
 10  period_end_utc    61367 non-null  object 
 11  c_by_hour         61367 non-null  int64  
 12  load_forecast_da  61367 non-null  float64
 13  load_actual       61367 non-null  float64
 14  off_wind_da       61367 non-null  float64
 15  off_wind_act      61367 non-null  float64
 16  on_wind_da        61367 non-null  float6

In [117]:

df_merged_h.to_csv("../../data_cleaned/merged/02_4_clean_data_rich_columns.csv", index=False)

In [118]:
df_merged_h.hour.values



array([ 0,  1,  2, ..., 20, 21, 22], dtype=int64)

In [119]:
df = df_merged_h.copy()
df = df.drop(columns=["week","load_actual","gen_actual", "off_wind_act", "on_wind_act", "solar_act", "res_sum_act","c_by_hour","interpolated","imputed"])

In [120]:
df_merged_h.to_csv("../../data_cleaned/merged/02_4_clean_data.csv", index=False)

In [121]:
df.columns

Index(['date', 'year', 'month', 'day', 'dayofyear', 'hour', 'dayofweek',
       'price', 'period_start_utc', 'period_end_utc', 'load_forecast_da',
       'off_wind_da', 'on_wind_da', 'solar_da', 'gen_forecast_da',
       'res_sum_da', 'dayofyear_sin1', 'dayofyear_cos1', 'hour_sin1',
       'hour_cos1', 'dayofweek_sin1', 'dayofweek_cos1', 'is_holiday',
       'day_type', 'price_gas'],
      dtype='object')

In [122]:
for i in sorted(df["hour"].dropna().unique()):
    df_temp_h = df[df["hour"] == i].copy()
    df_temp_h.to_csv(
        f"../../data_cleaned/merged_hour/02_4_clean_data_daily_hour_{int(i):02d}.csv",
        index=False
    )
